In [0]:
from pyspark import pipelines as dp

In [0]:
@dp.table(
    name="wanderbricks_employees_stream",
    table_properties={'quality':"bronze"}
)
def fn_employees_stream():
    df = spark.readStream.table('dev.sdp81.wander_employee')
    return df

In [0]:
@dp.materialized_view(
    name="wanderbricks_hosts_view",
    table_properties={'quality':"bronze"}
)
def fn_employees_stream():
    df = spark.read.table('dev.sdp81.wander_hosts')
    return df

In [0]:
@dp.table(
    name="wanderbricks_hosts_autoloader", table_properties={"quality": "bronze"}
)
def fn_wanderbricks_destination_autoloader():
    df = (
        spark.readStream.format("cloudFiles")
        .option(
            "cloudFiles.schemaHints",
            "host_id long,name string,email string,phone string,is_verified boolean,is_active boolean,rating float,country string,joined_at date"
        )
        .option(
            "cloudFiles.schemaLocation",
            "/Volumes/dev/sdp81/destinations_volume/autoloader/schemas/",
        )
        .option("cloudFiles.format", "csv")
        .load("/Volumes/dev/sdp81/destinations_volume/autoloader/files/")
    )
    return df

In [0]:
dp.create_streaming_table('wanderbricks_hosts_total')

@dp.append_flow(
    target="wanderbricks_hosts_total"
)
def fn_wanderbricks_autoloader():
    df = spark.readStream.table('wanderbricks_hosts_autoloader')
    return df

@dp.append_flow(
    target="wanderbricks_hosts_total"
)
def fn_wanderbricks_view():
    df = spark.readStream.table('wanderbricks_hosts_view')
    return df

In [0]:
@dp.table(
    name="wanderbricks_hosts_employees_joined",
    table_properties={"quality": "bronze"}
)
def fn_hosts_employees_joined():
    hosts_total = spark.readStream.table("dev.sdp81.wanderbricks_hosts_total").alias("h")
    employees_stream = spark.readStream.table("wanderbricks_employees_stream").alias("e")
    
    df = employees_stream.join(hosts_total, on="host_id", how="inner").select(
        "e.*",
        hosts_total["name"].alias("host_name"),
        hosts_total["email"].alias("host_email"),
        hosts_total["phone"].alias("host_phone"),
        hosts_total["country"].alias("host_country"),
        hosts_total["joined_at"].alias("host_joined_at")
    )
    return df